# pandas and SQL

## Overview
pandas integrates directly with SQL databases through `read_sql` (read) and `to_sql` (write). These functions bridge the DataFrame world and the relational world — essential for data pipelines that pull data from databases, transform it in Python, and write results back.

**Key functions:**

| Function | Purpose |
|---|---|
| `pd.read_sql(query, con)` | Execute a SQL query and return a DataFrame |
| `pd.read_sql_query(query, con)` | Same as read_sql for queries |
| `pd.read_sql_table(table, con)` | Read an entire table |
| `df.to_sql(name, con)` | Write a DataFrame to a SQL table |

**SQLAlchemy 2.0 usage:**
All `read_sql` and `to_sql` calls should pass an open `Connection` object (not the Engine directly) when using SQLAlchemy 2.0, to avoid deprecation warnings.

---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///:memory:", echo=False)

with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE patients (patient_id TEXT PRIMARY KEY, first_name TEXT,
            last_name TEXT, province TEXT, dob TEXT, sex TEXT)
    """))
    conn.execute(text("""
        CREATE TABLE encounters (enc_id INTEGER PRIMARY KEY, patient_id TEXT,
            enc_date TEXT, enc_type TEXT, cost REAL DEFAULT 0.0)
    """))
    conn.execute(text("""
        INSERT INTO patients VALUES
            ('P001','Aroha','Ngata','NB','1985-03-15','F'),
            ('P002','Liam','Chen','ON','1990-07-22','M'),
            ('P003','Fatima','Rashid','BC','1978-11-05','F'),
            ('P004','James','MacLeod','NB','2001-01-30','M'),
            ('P005','Maria','Santos','QC','1995-06-18','F')
    """))
    conn.execute(text("""
        INSERT INTO encounters VALUES
            (1,'P001','2023-04-10','outpatient', 250.0),
            (2,'P001','2023-07-15','inpatient', 4800.0),
            (3,'P002','2023-03-01','outpatient',  80.0),
            (4,'P003','2023-09-20','specialist',  450.0),
            (5,'P004','2023-11-05','outpatient',  120.0),
            (6,'P005','2023-06-12','inpatient',  3200.0)
    """))

print("Schema ready")


---
## read_sql: database to DataFrame

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///:memory:", echo=False)
with engine.begin() as conn:
    conn.execute(text("CREATE TABLE patients (patient_id TEXT PRIMARY KEY, first_name TEXT, last_name TEXT, province TEXT, dob TEXT, sex TEXT)"))
    conn.execute(text("CREATE TABLE encounters (enc_id INTEGER PRIMARY KEY, patient_id TEXT, enc_date TEXT, enc_type TEXT, cost REAL DEFAULT 0.0)"))
    conn.execute(text("INSERT INTO patients VALUES ('P001','Aroha','Ngata','NB','1985-03-15','F'),('P002','Liam','Chen','ON','1990-07-22','M'),('P003','Fatima','Rashid','BC','1978-11-05','F'),('P004','James','MacLeod','NB','2001-01-30','M'),('P005','Maria','Santos','QC','1995-06-18','F')"))
    conn.execute(text("INSERT INTO encounters VALUES (1,'P001','2023-04-10','outpatient',250.0),(2,'P001','2023-07-15','inpatient',4800.0),(3,'P002','2023-03-01','outpatient',80.0),(4,'P003','2023-09-20','specialist',450.0),(5,'P004','2023-11-05','outpatient',120.0),(6,'P005','2023-06-12','inpatient',3200.0)"))

# ── read_sql_query: SQL → DataFrame ──────────────────────────────
with engine.connect() as conn:
    df_patients = pd.read_sql_query(
        text("SELECT * FROM patients ORDER BY last_name"),
        conn
    )
print("Patients DataFrame:")
print(df_patients.to_string(index=False))
print(f"dtypes:\n{df_patients.dtypes}")

# ── Parameterised read_sql ────────────────────────────────────────
with engine.connect() as conn:
    df_nb = pd.read_sql_query(
        text("SELECT patient_id, first_name, last_name FROM patients WHERE province = :prov"),
        conn,
        params={"prov": "NB"}
    )
print(f"\nNB patients ({len(df_nb)} rows):")
print(df_nb.to_string(index=False))

# ── JOIN query → DataFrame ────────────────────────────────────────
with engine.connect() as conn:
    df_joined = pd.read_sql_query(text("""
        SELECT p.patient_id, p.first_name, p.province,
               COUNT(e.enc_id)   AS enc_count,
               SUM(e.cost)       AS total_cost,
               AVG(e.cost)       AS avg_cost
        FROM   patients  p
        JOIN   encounters e ON p.patient_id = e.patient_id
        GROUP  BY p.patient_id, p.first_name, p.province
        ORDER  BY total_cost DESC
    """), conn)
print("\nPatient encounter summary:")
print(df_joined.to_string(index=False))


---
## to_sql: DataFrame to database, chunked reads

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///:memory:", echo=False)
with engine.begin() as conn:
    conn.execute(text("CREATE TABLE patients (patient_id TEXT PRIMARY KEY, first_name TEXT, last_name TEXT, province TEXT, dob TEXT, sex TEXT)"))
    conn.execute(text("INSERT INTO patients VALUES ('P001','Aroha','Ngata','NB','1985-03-15','F'),('P002','Liam','Chen','ON','1990-07-22','M')"))

# ── to_sql: DataFrame → table ─────────────────────────────────────
df_new = pd.DataFrame({
    "patient_id": ["P003","P004","P005"],
    "first_name":  ["Fatima","James","Maria"],
    "last_name":   ["Rashid","MacLeod","Santos"],
    "province":    ["BC","NB","QC"],
    "dob":         ["1978-11-05","2001-01-30","1995-06-18"],
    "sex":         ["F","M","F"],
})
with engine.begin() as conn:
    df_new.to_sql(
        "patients",
        conn,
        if_exists="append",    # 'fail' | 'replace' | 'append'
        index=False,           # do not write DataFrame index as a column
        method="multi",        # batch INSERT for better performance
    )
with engine.connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM patients")).scalar()
print(f"After to_sql append: {total} patients")

# ── Chunked read for large tables ─────────────────────────────────
print("\nChunked read (chunksize=2):")
with engine.connect() as conn:
    chunks = pd.read_sql_query(
        text("SELECT * FROM patients ORDER BY patient_id"),
        conn,
        chunksize=2    # returns an iterator of DataFrames
    )
    for i, chunk in enumerate(chunks):
        print(f"  Chunk {i+1}: {list(chunk['patient_id'])}")

# ── Data transformation pipeline: SQL → transform → SQL ──────────
print("\nTransformation pipeline:")
with engine.connect() as conn:
    df = pd.read_sql_query(text("SELECT * FROM patients"), conn)

# Transform in pandas
df["full_name"]   = df["first_name"] + " " + df["last_name"]
df["is_atlantic"] = df["province"].isin(["NB","NS","PE","NL"])
df_summary = (df.groupby("province", as_index=False)
                .agg(patient_count=("patient_id","count"),
                     atlantic=("is_atlantic","any")))

with engine.begin() as conn:
    df_summary.to_sql("province_summary", conn, if_exists="replace", index=False)
    result = pd.read_sql_query(text("SELECT * FROM province_summary"), conn)
print(result.to_string(index=False))


---
## Common Pitfalls

**1. read_sql with a plain string in SQLAlchemy 2.0**
`pd.read_sql('SELECT * FROM patients', engine)` raises a warning or error in SQLAlchemy 2.0 because raw string SQL is not directly executable. Wrap it: `pd.read_sql(text('SELECT * FROM patients'), conn)` inside a `with engine.connect()` block, or use `pd.read_sql_table()` / `pd.read_sql_query()`.

**2. to_sql with if_exists='replace' dropping all indexes and constraints**
`df.to_sql('patients', engine, if_exists='replace')` drops and recreates the table with no indexes, no PRIMARY KEY, and an auto-added integer index column. For production tables, use `if_exists='append'` with pre-created schema, or write a custom upsert instead.

**3. Numeric type mismatch: float64 vs REAL vs DECIMAL**
pandas reads SQL REAL/FLOAT as float64. SQL DECIMAL/NUMERIC may become object dtype if precision is high. Always inspect `df.dtypes` after `read_sql` and cast explicitly. Use `dtype` parameter in `to_sql` to control the SQL column type written.

**4. read_sql loading entire large tables into memory**
A `SELECT * FROM events` on a 100 million-row table loads all rows into a single DataFrame. Use `chunksize` to process in batches: `for chunk in pd.read_sql(..., chunksize=10000):`.

**5. NULL handling differences between pandas and SQL**
SQL NULLs become `NaN` in float columns and `None` in object columns in pandas. Comparisons like `df['col'] == None` never match (use `df['col'].isna()`). When writing back with `to_sql`, pandas NaN is written as SQL NULL.

**6. to_sql performance with default row-by-row inserts**
By default `to_sql` inserts one row per statement. For large DataFrames this is extremely slow. Use `method='multi'` (batch INSERT) or, for PostgreSQL, `method=psycopg2_copy_from` for COPY-speed bulk loads.


---
*sql_methods_library - Samantha McGarrigle*